# Full Pipeline: Custom EHR → FHIR R4

The same standardize pipeline aimed at FHIR R4, ending in a
transaction **Bundle** and **NDJSON** export.

> **Prerequisites** — `pip install "portiere-health[polars]"`. This notebook is
> **fully offline**: no model downloads, no network calls, no API keys. It uses
> the bundled ICD-10-CM / LOINC / RxNorm vocabulary subset and BM25 lexical
> retrieval, and asserts the no-egress guarantee with `offline=True`.
> Every cell is deterministic — run top to bottom.

In [1]:
import json
import tempfile
from pathlib import Path

import polars as pl

import portiere
from portiere import __version__

WORK = Path(tempfile.mkdtemp(prefix="portiere_nb_"))
print(f"portiere v{__version__} — working dir: {WORK}")

portiere v0.4.0 — working dir: /var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_ecckgfrs


## Step 1 — Source data

The same kind of messy EHR export as the OMOP walkthrough — this time we
standardize it to **FHIR R4** resources instead of OMOP tables. The pipeline is
identical; only `target_model` changes.

In [2]:
import random

random.seed(7)
src_dir = WORK / "ehr_export"
src_dir.mkdir(parents=True)

n = 40
observations = pl.DataFrame({
    "patient_id": [f"P{random.randint(1, 20):05d}" for _ in range(n)],
    "lab_code":   [random.choice(["2160-0", "2345-7", "718-7", "4548-4"]) for _ in range(n)],
    "lab_value":  [round(random.uniform(3.5, 9.5), 1) for _ in range(n)],
    "lab_unit":   ["mmol/L"] * n,
    "lab_date":   [f"2024-{random.randint(1, 12):02d}-{random.randint(1, 28):02d}"
                   for _ in range(n)],
})
observations.write_csv(src_dir / "lab_results.csv")
print(observations.head(4))

shape: (4, 5)
┌────────────┬──────────┬───────────┬──────────┬────────────┐
│ patient_id ┆ lab_code ┆ lab_value ┆ lab_unit ┆ lab_date   │
│ ---        ┆ ---      ┆ ---       ┆ ---      ┆ ---        │
│ str        ┆ str      ┆ f64       ┆ str      ┆ str        │
╞════════════╪══════════╪═══════════╪══════════╪════════════╡
│ P00011     ┆ 718-7    ┆ 3.9       ┆ mmol/L   ┆ 2024-05-01 │
│ P00005     ┆ 2345-7   ┆ 4.1       ┆ mmol/L   ┆ 2024-03-14 │
│ P00013     ┆ 2160-0   ┆ 5.1       ┆ mmol/L   ┆ 2024-09-12 │
│ P00002     ┆ 2345-7   ┆ 7.7       ┆ mmol/L   ┆ 2024-10-19 │
└────────────┴──────────┴───────────┴──────────┴────────────┘


In [3]:
# Offline configuration: BM25 retrieval over the bundled vocabulary subset,
# no embedding model (nothing to download), no reranker, and offline=True —
# construction FAILS if any provider could send data off this machine.
from portiere._demo_data import vocabulary_dir
from portiere.config import (
    EmbeddingConfig,
    KnowledgeLayerConfig,
    PortiereConfig,
    RerankerConfig,
)
from portiere.knowledge import build_knowledge_layer

knowledge_paths = build_knowledge_layer(
    athena_path=str(vocabulary_dir()),
    output_path=str(WORK / "knowledge_index"),
    backend="bm25s",
    vocabularies=["ICD10CM", "LOINC", "RxNorm"],
)

config = PortiereConfig(
    local_project_dir=WORK / "project",
    knowledge_layer=KnowledgeLayerConfig(backend="bm25s", **knowledge_paths),
    embedding=EmbeddingConfig(provider="none"),
    reranker=RerankerConfig(provider="none", model=""),
    offline=True,  # the executable no-egress guarantee (v0.4.0)
)
print("offline mode asserted:", config.offline)

2026-07-24 10:11:00 [info     ] athena.loading_concepts        path=/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/src/portiere/_demo_data/vocabulary/CONCEPT.csv


2026-07-24 10:11:00 [info     ] athena.concepts_loaded         count=27 vocabularies=['ICD10CM', 'LOINC', 'RxNorm']


2026-07-24 10:11:00 [info     ] athena.bm25s_built             count=27 path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_ecckgfrs/knowledge_index/concepts.json


2026-07-24 10:11:00 [info     ] athena.knowledge_layer_built   backend=bm25s concepts=27 paths={'bm25s_corpus_path': '/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_ecckgfrs/knowledge_index/concepts.json'}


offline mode asserted: True


## Step 2 — Point the pipeline at FHIR R4

`target_model="fhir_r4"` swaps the whole target universe: schema mapping now
aims at FHIR resources/elements (Portiere ships 18 R4 resources — see the
support matrix in `20-multi-standard-support.md`).

In [4]:
project = portiere.init(
    name="ehr-to-fhir",
    target_model="fhir_r4",
    vocabularies=["LOINC"],
    config=config,
)

src_obs = project.add_source(str(src_dir / "lab_results.csv"), name="lab_results")
print(f"rows: {src_obs['row_count']}, code columns: {src_obs.get('code_columns')}")

2026-07-24 10:11:00 [info     ] PolarsEngine initialized      


2026-07-24 10:11:00 [debug    ] local_storage.initialized      base_dir=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_ecckgfrs/project


2026-07-24 10:11:00 [info     ] local_storage.project_created  name=ehr-to-fhir project_id=109aff90-89cb-4878-af9d-1b7197d6cf2b


2026-07-24 10:11:00 [info     ] project.source_added           project=ehr-to-fhir source=lab_results


2026-07-24 10:11:00 [warning  ] manifest.git_dirty             hint='Working tree has uncommitted changes; manifest records git_dirty=true' project_root=/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/docs/notebooks_examples


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-07-24 10:11:03 [info     ] gx_profiler.profiled           columns=5 rows=40 source=lab_results


2026-07-24 10:11:03 [info     ] project.profiled               source=lab_results


rows: 40, code columns: None


In [5]:
schema_map = project.map_schema(src_obs)
for item in schema_map.items:
    print(f"  {item.source_column:10s} -> "
          f"{(item.effective_target_table or '?'):16s}."
          f"{(item.effective_target_column or '?'):24s} "
          f"conf={item.confidence:.2f} [{item.status}]")

2026-07-24 10:11:03 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=none target_model=fhir_r4


2026-07-24 10:11:03 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=none target_standard=fhir_r4


2026-07-24 10:11:03 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=none


2026-07-24 10:11:03 [info     ] local_schema_mapper.computing_target_embeddings standard=fhir_r4 targets=168


2026-07-24 10:11:03 [info     ] local_schema_mapper.initialized standard=fhir_r4


2026-07-24 10:11:03 [info     ] Stage 2 complete               auto=4 review=0 total=5


2026-07-24 10:11:03 [info     ] local_storage.schema_mapping_saved items_count=5 project=ehr-to-fhir


2026-07-24 10:11:03 [info     ] project.schema_mapped          auto=4 project=ehr-to-fhir total=5


  patient_id -> Patient         .id                       conf=0.95 [MappingStatus.AUTO_ACCEPTED]
  lab_code   -> Observation     .code                     conf=0.95 [MappingStatus.AUTO_ACCEPTED]
  lab_value  -> Observation     .valueQuantity            conf=0.95 [MappingStatus.AUTO_ACCEPTED]
  lab_unit   -> Observation     .valueQuantity            conf=0.50 [MappingStatus.UNMAPPED]
  lab_date   -> Observation     .effectiveDateTime        conf=0.95 [MappingStatus.AUTO_ACCEPTED]


In [6]:
concept_map = project.map_concepts(source=src_obs, code_columns=["lab_code"])
for item in concept_map.items:
    print(f"  {item.source_code:8s} -> {str(item.target_concept_id):10s} "
          f"{(item.target_concept_name or '')[:44]}")

2026-07-24 10:11:03 [info     ] Stage 3: Concept mapping (local) columns=['lab_code'] knowledge_backend=bm25s


2026-07-24 10:11:03 [info     ] Mapping column: lab_code      


2026-07-24 10:11:03 [info     ] knowledge.creating_backend     backend=bm25s path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_ecckgfrs/knowledge_index/concepts.json


Split strings:   0%|          | 0/27 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/27 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-24 10:11:03 [info     ] Stage 3 complete               auto_rate=0.0% total_codes=4


2026-07-24 10:11:03 [info     ] local_storage.concept_mapping_saved items_count=4 project=ehr-to-fhir


2026-07-24 10:11:03 [info     ] project.concepts_mapped        auto_rate=0.0 project=ehr-to-fhir total=4


  718-7    -> None       
  4548-4   -> None       
  2160-0   -> None       
  2345-7   -> None       


## Step 3 — ETL to FHIR-shaped output

In [7]:
etl_out = WORK / "fhir_output"
result = project.run_etl(src_obs, output_dir=str(etl_out),
                         schema_mapping=schema_map, concept_mapping=concept_map)
tables = sorted(etl_out.rglob("*.csv"))
for f in tables:
    print(f"  {f.relative_to(etl_out)}")

2026-07-24 10:11:03 [info     ] Reading source                 format=csv path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_ecckgfrs/ehr_export/lab_results.csv


2026-07-24 10:11:03 [info     ] Processing table               columns=1 table=Patient


2026-07-24 10:11:03 [info     ] Processing table               columns=3 table=Observation


2026-07-24 10:11:03 [info     ] ETL complete                   duration=0.003094 success=True


2026-07-24 10:11:03 [info     ] project.etl_complete           output_dir=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_ecckgfrs/fhir_output project=ehr-to-fhir


  Observation.csv
  Patient.csv


## Step 4 — Assemble FHIR resources, Bundle, and NDJSON

The ETL emits flattened per-resource tables. `portiere.export.fhir` turns
resource dicts into a **transaction Bundle** (POST-ready) or **NDJSON** files
(bulk-import-ready). Validation against US Core / mCODE profiles:
`portiere validate --fhir-profile us-core-6.1.0`.

In [8]:
from portiere.export.fhir.bundle import to_transaction_bundle
from portiere.export.fhir.ndjson import to_ndjson_files

# Build Observation resources from the ETL output (flattened rows -> dicts).
# Falls back to the source rows if a table is empty, so the export step is
# always demonstrable.
etl_rows = []
for f in tables:
    etl_rows.extend(pl.read_csv(f, infer_schema_length=0).to_dicts())
if not etl_rows:
    etl_rows = observations.to_dicts()

resources = []
for i, row in enumerate(etl_rows[:10]):
    resources.append({
        "resourceType": "Observation",
        "id": f"obs-{i}",
        "status": "final",
        **{k: str(v) for k, v in row.items() if v not in (None, "")},
    })

bundle = to_transaction_bundle(resources)
print(f"Bundle: type={bundle['type']}, entries={len(bundle['entry'])}")

nd_paths = to_ndjson_files(resources, out_dir=WORK / "ndjson")
for p in nd_paths:
    print("NDJSON:", p.name, "-", sum(1 for _ in open(p)), "resources")

Bundle: type=transaction, entries=10
NDJSON: Observation.ndjson - 10 resources


In [9]:
# Peek at one Bundle entry
print(json.dumps(bundle["entry"][0], indent=2)[:600])

{
  "fullUrl": "urn:uuid:5da4fc96-6692-453b-af6c-78843ca2d43a",
  "resource": {
    "resourceType": "Observation",
    "id": "obs-0",
    "status": "final",
    "code": "718-7",
    "valueQuantity": "3.9",
    "effectiveDateTime": "2024-05-01"
  },
  "request": {
    "method": "POST",
    "url": "Observation"
  }
}
